# Flow Feature Engineering

This notebook converts packet-level network traffic data into flow-level features suitable for machine learning analysis. Flow statistics capture behavioural patterns within network communications and provide more meaningful inputs for intrusion detection models.

## Import Required Libraries

In [15]:
# Libraries for data manipulation and numerical operations
import pandas as pd
import numpy as np

## Load Processed Dataset

The cleaned and labelled packet dataset generated in the previous stage is loaded for feature engineering.

In [16]:
# Load labelled packet dataset
df = pd.read_csv("../data/processed/dataset_v3_labeled.csv")

# Display dataset shape
print(df.shape)

# Inspect first rows
df.head()

(15331, 9)


,frame.time_epoch,ip.src,ip.dst,udp.srcport,udp.dstport,frame.len,frame.time_delta,multi_label,binary_label
0,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,156,0.000000,0,0
1,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,155,1.625913,0,0
2,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,157,1.975084,0,0
3,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,185,1.862526,0,0
4,1.774143e+09,192.168.56.10,192.168.56.15,47927,5005,172,2.117311,0,0


## Handling Missing Port Values

Some packets may not contain port information. Missing values are replaced with 0 to ensure consistent flow identification.

In [17]:
# Replace missing port values with 0 
df["udp.srcport"] = df["udp.srcport"].fillna(0) 
df["udp.dstport"] = df["udp.dstport"].fillna(0)

## Time Window Creation

Packets are grouped into fixed 10-second time windows to segment the traffic into manageable behavioural units.

In [18]:
# Create 10-second time windows using integer division
df["time_window"] = (df["frame.time_epoch"] // 10).astype(int)

## Flow Definition

Flows are defined using a combination of:

Source IP
Destination IP
Source port
Destination port
Time window

In [19]:
# Define flows using endpoints + time window
df["flow_id"] = (
    df["ip.src"] + "_" +
    df["ip.dst"] + "_" +
    df["udp.srcport"].astype(str) + "_" +
    df["udp.dstport"].astype(str) + "_" +
    df["time_window"].astype(str)
)

## Flow Features Computation

For each flow, several statistical features are calculated to represent communication behaviour.
| Feature           | Description                     |
| ----------------- | ------------------------------- |
| packet_count      | packets in flow                 |
| total_bytes       | total transmitted bytes         |
| avg_packet_size   | mean packet size                |
| packet_rate       | packets per second              |
| flow_duration     | flow lifetime                   |
| interarrival_mean | mean packet interval            |
| interarrival_std  | variability of packet intervals |


In [20]:
# Group packets by flow identifier
flows = df.groupby("flow_id")


## Flow Features Generation

Several statistical features are calculated to describe the behaviour of each network flow. These features summarise packet-level activity into behavioural characteristics that can be used by machine learning models.

Flows containing only a single packet cannot produce a valid standard deviation for packet inter-arrival times. In these cases, missing values are replaced with 0 to maintain a consistent dataset for model training.

In [21]:
# Create dataframe to store flow-level features
flow_features = pd.DataFrame()

# Number of packets within each flow
flow_features["packet_count"] = flows.size()

# Total number of bytes transmitted within each flow
flow_features["total_bytes"] = flows["frame.len"].sum()

# Average packet size for the flow
flow_features["avg_packet_size"] = flows["frame.len"].mean()

# Flow duration calculated as the difference between the first and last packet timestamps
flow_features["flow_duration"] = ( flows["frame.time_epoch"].max() - flows["frame.time_epoch"].min() )

# Mean time between packets within the flow
flow_features["interarrival_mean"] = flows["frame.time_delta"].mean()

# Standard deviation of packet inter-arrival times
flow_features["interarrival_std"] = flows["frame.time_delta"].std()

# Replace NaN values produced by single-packet flows
# Single packet flows cannot have a standard deviation
flow_features["interarrival_std"] = flow_features["interarrival_std"].fillna(0)

### Packet Rate

Packet rate represents the number of packets transmitted per second within a flow.

In [22]:
# Replace zero duration values to avoid division errors
flow_features["flow_duration"] = flow_features["flow_duration"].replace(0, 0.001)

# Calculate packets per second within each flow
flow_features["packet_rate"] = ( flow_features["packet_count"] / flow_features["flow_duration"] )

# Log transform packet count (helps reduce skew)
flow_features["packet_count_log"] = np.log1p(flow_features["packet_count"])

### Log Transformation of Packet Rate
To further stabilise feature scaling, a log transformation is also applied to packet count.

In [23]:
# Apply logarithmic transformation to packet rate
flow_features["packet_rate_log"] = np.log1p(flow_features["packet_rate"])

## Flow Label Assignment

Flow labels are derived from packet-level labels. The maximum label value within each flow is used.

This ensures that if any packet within a flow is malicious, the entire flow is labelled as malicious.

In [24]:
# Assign binary labels (0 = normal, 1 = attack)
flow_features["binary_label"] = flows["binary_label"].max()

# Assign multiclass labels (0 = normal, 1 = spoof, 2 = timing, 3 = flood)
flow_features["multi_label"] = flows["multi_label"].max()

## Reset Dataset Index

The flow identifier is converted from index to a column to maintain a clean dataset structure.

In [25]:
# Reset index to make flow_id a column
flow_features = flow_features.reset_index()

## Flow Dataset Verification

A quick inspection is performed to verify that the feature generation process worked correctly and that all classes are represented in the dataset.

In [26]:
# Preview first rows of flow dataset
flow_features.head()

,flow_id,packet_count,total_bytes,avg_packet_size,flow_duration,interarrival_mean,interarrival_std,packet_rate,packet_count_log,packet_rate_log,binary_label,multi_label
0,192.168.56.10_192.168.56.15_35318_5005_177414451,2,327,163.500000,1.481448,7.437044,8.422485,1.350031,1.098612,0.854428,0,0
1,192.168.56.10_192.168.56.15_35318_5005_177414452,6,1005,167.500000,8.954694,1.752229,0.264699,0.670040,1.945910,0.512847,0,0
2,192.168.56.10_192.168.56.15_35318_5005_177414453,6,1004,167.333333,7.954704,1.694258,0.555134,0.754271,1.945910,0.562053,0,0
3,192.168.56.10_192.168.56.15_35318_5005_177414454,6,1005,167.500000,8.264359,1.612960,0.418324,0.726009,1.945910,0.545812,0,0
4,192.168.56.10_192.168.56.15_35318_5005_177414455,6,1058,176.333333,8.157557,1.676504,0.543603,0.735514,1.945910,0.551304,0,0


In [27]:
# Check binary class distribution
print("Binary distribution:")
print(flow_features["binary_label"].value_counts())

Binary distribution:
binary_label
0    227
1    142
Name: count, dtype: int64


In [28]:
# Check multiclass distribution
print("\nMulticlass distribution:")
print(flow_features["multi_label"].value_counts())


Multiclass distribution:
multi_label
0    227
1     51
2     51
3     40
Name: count, dtype: int64


## Flow Dataset Output

The generated flow-level dataset is saved for use in the machine learning training stage.

In [29]:
# Save processed flow dataset
flow_features.to_csv("../data/processed/flows_v3.csv", index=False)